In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image
import torchvision.transforms as T

In [2]:
class EmbeddingNet(nn.Module):
    def __init__(self):
        super(EmbeddingNet, self).__init__()
        self.convnet = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 320x320

            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 160x160

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 80x80

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 40x40

            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(512, 128)

    def forward(self, x):
        x = self.convnet(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)  # batch_size, 128)
        return x
    
transform = T.Compose([
    T.Resize((640, 640)),
    T.ToTensor(),
])

In [4]:
embedding_net = EmbeddingNet()
embedding_net.load_state_dict(torch.load("embedding_net.pth", map_location="cpu"))
embedding_net.eval() 

EmbeddingNet(
  (convnet): Sequential(
    (0): Conv2d(3, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (fc): Linear(in_features=512, out_features=128, bias=True)
)

In [10]:
def evaluate_anchor(anchor_path, positive_dir, negative_dir, model, device="cpu"):
    model = model.to(device)
    model.eval()

    # Anchor embedding
    anchor_img = Image.open(anchor_path).convert("RGB")
    anchor_tensor = transform(anchor_img).unsqueeze(0).to(device)
    with torch.no_grad():
        anchor_embed = model(anchor_tensor)

    def compute_distances(img_dir):
        distances = []
        files = []
        for fname in os.listdir(img_dir):
            fpath = os.path.join(img_dir, fname)
            if not os.path.isfile(fpath):
                continue
            img = Image.open(fpath).convert("RGB")
            img_tensor = transform(img).unsqueeze(0).to(device)
            with torch.no_grad():
                embed = model(img_tensor)
                dist = F.pairwise_distance(anchor_embed, embed).item()
                distances.append(dist)
                files.append(fname)
        return distances, files

    pos_distances, pos_files = compute_distances(positive_dir)
    neg_distances, neg_files = compute_distances(negative_dir)

    def summarize(distances, files):
        if not distances:
            return None
        distances = np.array(distances)
        min_idx = np.argmin(distances)
        max_idx = np.argmax(distances)
        return {
            "min": float(distances[min_idx]),
            "min_file": files[min_idx],
            "max": float(distances[max_idx]),
            "max_file": files[max_idx],
            "mean": float(np.mean(distances)),
        }

    results = {
        "positive": summarize(pos_distances, pos_files),
        "negative": summarize(neg_distances, neg_files),
    }

    return results


# Example usage
anchor_path = "triplet_data/anchor/1.jpg"
positive_dir = "triplet_data/positive"
negative_dir = "triplet_data/negative"

results = evaluate_anchor(anchor_path, positive_dir, negative_dir, embedding_net)

print("Positive distances:", results["positive"])
print("Negative distances:", results["negative"])

Positive distances: {'min': 1.131371027440764e-05, 'min_file': '72.jpg', 'max': 5.5930681228637695, 'max_file': '9.jpg', 'mean': 1.3194939578697449}
Negative distances: {'min': 0.26026225090026855, 'min_file': '47.jpg', 'max': 21.660207748413086, 'max_file': '90.jpg', 'mean': 6.309079758524895}
